# HyperLogistics-Stream: End-to-End Pipeline on Google Colab Pro

**Course:** COMP-SCI 5540 Principles of Big Data Management — UMKC Fall 2026  
**Authors:** Tony Nguyen & Joe Doan  

**Runtime:** Colab Pro, High-RAM (≥ 25 GB), Standard CPU  
**Runtime type:** `Python 3` — no GPU needed

### Execution Order
1. **Section 0** — Environment setup (Java, Kafka binary, packages)
2. **Section 1** — Mount Drive, clone repo, configure paths
3. **Section 2** — Download BTS + NOAA data
4. **Section 3** — EDA (optional, reuses `scripts/run_eda.py`)
5. **Section 4** — Phase 1: Offline batch training (GBT + RF)
6. **Section 5** — Phase 2+3: Kafka producer + Spark Structured Streaming
7. **Section 6** — Phase 5: FastAPI (background) + Streamlit via ngrok
8. **Section 7** — RQ2 Benchmarks
9. **Section 8** — Analysis figures

> **Tip:** Run cells sequentially. Each section prints status lines — wait for `===` completion markers before moving on.

---
## Section 0 — Environment Setup

Installs Java 11, PySpark 3.5, confluent-kafka, and starts a Kafka broker directly in the Colab runtime (no Docker required).

In [ ]:
# Cell 0.1 — Install Java 11 + Python packages
import subprocess, sys

print("Installing Java 11...")
subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-11-jdk-headless"], check=True)

print("Installing Python packages...")
subprocess.run([
    sys.executable, "-m", "pip", "install", "-q",
    "pyspark==3.5.0",
    "confluent-kafka>=2.3.0",
    "fastapi>=0.111.0",
    "uvicorn[standard]>=0.29.0",
    "pydeck>=0.9.0",
    "plotly>=5.20.0",
    "pyarrow>=14.0.0",
    "pyyaml>=6.0",
    "tqdm>=4.66.0",
    "tabulate>=0.9.0",
    "tzdata>=2024.1",
    "nest-asyncio>=1.6.0",
    "pyngrok>=7.0.0",
], check=True)

import os
# Set JAVA_HOME for PySpark
java_home = subprocess.check_output(
    "readlink -f $(which java) | sed 's|/bin/java||'", shell=True
).decode().strip()
os.environ["JAVA_HOME"] = java_home
print(f"JAVA_HOME={java_home}")

import pyspark
print(f"PySpark {pyspark.__version__} ready")

In [ ]:
# Cell 0.2 — Download + start Kafka 3.7 binary (no Docker)
import subprocess, time, os

KAFKA_VERSION = "3.7.0"
SCALA_VERSION = "2.13"
KAFKA_DIR = f"/opt/kafka_{SCALA_VERSION}-{KAFKA_VERSION}"
KAFKA_BIN = f"{KAFKA_DIR}/bin"
KAFKA_TARBALL = f"/tmp/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"
KAFKA_URL = f"https://downloads.apache.org/kafka/{KAFKA_VERSION}/kafka_{SCALA_VERSION}-{KAFKA_VERSION}.tgz"

if not os.path.exists(KAFKA_DIR):
    print(f"Downloading Kafka {KAFKA_VERSION}...")
    subprocess.run(["wget", "-q", KAFKA_URL, "-O", KAFKA_TARBALL], check=True)
    subprocess.run(["tar", "-xzf", KAFKA_TARBALL, "-C", "/opt"], check=True)
    print("Kafka extracted.")
else:
    print("Kafka already extracted.")

# Tune server.properties for Colab
server_props = f"{KAFKA_DIR}/config/server.properties"
with open(server_props, "a") as f:
    f.write("\n# HyperLogistics-Stream\n")
    f.write("message.max.bytes=10485760\n")
    f.write("log.retention.hours=2\n")
    f.write("num.partitions=8\n")
    f.write("auto.create.topics.enable=true\n")

# Start Zookeeper
print("Starting Zookeeper...")
subprocess.Popen(
    f"{KAFKA_BIN}/zookeeper-server-start.sh {KAFKA_DIR}/config/zookeeper.properties",
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(5)

# Start Kafka broker
print("Starting Kafka broker...")
subprocess.Popen(
    f"{KAFKA_BIN}/kafka-server-start.sh {KAFKA_DIR}/config/server.properties",
    shell=True, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL
)
time.sleep(8)

# Create topic
result = subprocess.run([
    f"{KAFKA_BIN}/kafka-topics.sh",
    "--bootstrap-server", "localhost:9092",
    "--create", "--if-not-exists",
    "--topic", "flights_stream",
    "--partitions", "8",
    "--replication-factor", "1"
], capture_output=True, text=True)
print(result.stdout or result.stderr)

# Verify
verify = subprocess.run(
    [f"{KAFKA_BIN}/kafka-topics.sh", "--bootstrap-server", "localhost:9092", "--list"],
    capture_output=True, text=True
)
print(f"Topics: {verify.stdout.strip()}")
print("Kafka ready on localhost:9092")

---
## Section 1 — Mount Drive & Configure Paths

In [ ]:
# Cell 1.1 — Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
# Change this to match your Drive folder name
DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f"Drive root: {DRIVE_ROOT}")

In [ ]:
# Cell 1.2 — Clone / pull repo into /content
import os, subprocess

REPO_URL = "https://github.com/YOUR_GITHUB_USERNAME/HyperLogistic_Stream.git"  # <-- update this
REPO_DIR = "/content/HyperLogistic_Stream"

if os.path.exists(REPO_DIR):
    subprocess.run(["git", "-C", REPO_DIR, "pull"], check=True)
    print("Repo updated.")
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)
    print("Repo cloned.")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
# Cell 1.3 — Create data directories on Drive
import os

DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"
dirs = [
    f"{DRIVE_ROOT}/data/raw/bts",
    f"{DRIVE_ROOT}/data/raw/noaa",
    f"{DRIVE_ROOT}/data/bronze",
    f"{DRIVE_ROOT}/data/silver",
    f"{DRIVE_ROOT}/data/gold",
    f"{DRIVE_ROOT}/models",
    f"{DRIVE_ROOT}/benchmark/results",
    "/content/checkpoints",
]
for d in dirs:
    os.makedirs(d, exist_ok=True)
    print(f"  {d}")
print("Directories ready.")

In [ ]:
# Cell 1.4 — Point pipeline at Colab config
import os
# Override the default config path so all pipeline modules use Colab paths
os.environ["HYPERLOGISTICS_CONFIG"] = "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml"
print("Config: pipeline_config_colab.yaml (Drive paths, localhost:9092 Kafka)")

---
## Section 2 — Download Data

BTS On-Time Performance (2018–2024) + NOAA ISD hourly weather.  
Total download: ~8–10 GB. Files are stored on Drive so they persist across sessions.

In [ ]:
# Cell 2.1 — Download BTS data
# Download all months for training years + 2024 streaming year.
# Skips files already on Drive automatically.

DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"
BTS_DIR = f"{DRIVE_ROOT}/data/raw/bts"

!python scripts/download_bts.py \
    --years 2018 2019 2021 2022 2023 2024 \
    --output-dir "{BTS_DIR}" \
    --extract

In [ ]:
# Cell 2.2 — Download NOAA weather
DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"
NOAA_DIR = f"{DRIVE_ROOT}/data/raw/noaa"

!python scripts/download_noaa.py \
    --years 2018 2019 2021 2022 2023 2024 \
    --output-dir "{NOAA_DIR}"

In [ ]:
# Cell 2.3 — Check data sizes
import os
DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"

for subdir in ["data/raw/bts", "data/raw/noaa"]:
    path = f"{DRIVE_ROOT}/{subdir}"
    total_bytes = sum(
        os.path.getsize(os.path.join(dp, f))
        for dp, _, files in os.walk(path)
        for f in files
    )
    count = sum(len(files) for _, _, files in os.walk(path))
    print(f"  {subdir}: {count} files, {total_bytes/1e9:.2f} GB")

---
## Section 3 — EDA (Optional)

In [ ]:
# Cell 3.1 — Run EDA on January 2024 sample
import os
DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"

# Point run_eda.py at the Colab paths by patching the main() call
import sys
sys.path.insert(0, '/content/HyperLogistic_Stream')

from scripts.run_eda import clean_bts_data, clean_noaa_weather, perform_stream_join, generate_visualizations

bts_csv = f"{DRIVE_ROOT}/data/raw/bts/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2024_1.csv"
noaa_dir = f"{DRIVE_ROOT}/data/raw/noaa/2024"

flights = clean_bts_data(bts_csv, max_rows=100000)
weather = clean_noaa_weather(noaa_dir)
joined = perform_stream_join(flights, weather)
generate_visualizations(joined, "/content/HyperLogistic_Stream/reports/figures")

print(f"Joined sample: {len(joined):,} flights")

In [ ]:
# Cell 3.2 — Display EDA figures inline
from IPython.display import Image, display
from pathlib import Path

figs_dir = Path("/content/HyperLogistic_Stream/reports/figures")
for fig in sorted(figs_dir.glob("fig[1-5]*.png")):
    print(f"\n--- {fig.name} ---")
    display(Image(str(fig), width=900))

---
## Section 4 — Phase 1: Offline Batch Training

Trains GBT and RF regressors on BTS 2018–2022, validates on 2023.  
Expected runtime on Colab Pro High-RAM: **20–45 minutes** depending on data volume.  
Models are saved to Google Drive and reused by the streaming pipeline.

In [ ]:
# Cell 4.1 — Build SparkSession for batch training
import os
os.environ["HYPERLOGISTICS_CONFIG"] = "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml"

from pyspark.sql import SparkSession

# Colab Pro High-RAM: use all available cores, tune memory
spark_train = (
    SparkSession.builder
    .appName("HyperLogistics-BatchTrain")
    .master("local[*]")
    .config("spark.driver.memory", "20g")
    .config("spark.executor.memory", "20g")
    .config("spark.sql.shuffle.partitions", "64")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.sql.parquet.compression.codec", "snappy")
    .config("spark.ui.port", "4040")
    .getOrCreate()
)
spark_train.sparkContext.setLogLevel("WARN")
print(f"Spark {spark_train.version} started. UI: http://localhost:4040")

In [ ]:
# Cell 4.2 — Run batch training pipeline
# This cell imports the training functions directly to avoid subprocess overhead.

import sys, os, time, json
sys.path.insert(0, '/content/HyperLogistic_Stream')
os.environ["HYPERLOGISTICS_CONFIG"] = "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml"

from pathlib import Path
import yaml
from pyspark.sql import functions as F
from pyspark.ml import Pipeline as SparkPipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import GBTRegressor, RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator

# Import pipeline helpers
from pipeline.batch_train_helpers import (
    load_bts_csv, add_utc_timestamps, load_noaa_weather, join_weather
)

# --- OR: run as subprocess if helpers are not importable ---
# Uncomment the block below and comment out the import above if needed:
# import subprocess
# result = subprocess.run([
#     sys.executable, "pipeline/01_batch_train.py",
#     "--shuffle-partitions", "64"
# ], capture_output=False)

print("Training pipeline imported. Loading data...")

In [ ]:
# Cell 4.3 — Run 01_batch_train.py as subprocess (recommended approach)
import subprocess, sys

print("Starting batch training (this will take ~20-45 min)...")
print("Monitor: http://localhost:4040 (Spark UI)")

result = subprocess.run(
    [
        sys.executable, "/content/HyperLogistic_Stream/pipeline/01_batch_train.py",
        "--shuffle-partitions", "64",
        # --skip-rf if you want GBT only and want to save ~15 min
    ],
    cwd="/content/HyperLogistic_Stream",
    env={
        **__import__('os').environ,
        "HYPERLOGISTICS_CONFIG": "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml",
    }
)

if result.returncode == 0:
    print("\n=== Training complete! ===")
else:
    print(f"Training exited with code {result.returncode}")

In [ ]:
# Cell 4.4 — Show training metrics
import json
DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"

metrics_path = f"{DRIVE_ROOT}/models/training_metrics.json"
with open(metrics_path) as f:
    metrics = json.load(f)

print("=== Training Metrics (RQ1) ===")
print(f"Train records : {metrics['train_count']:,}")
print(f"Val records   : {metrics['val_count']:,}")
print(f"GBT MAE       : {metrics['gbt_mae']:.3f} min")
print(f"GBT RMSE      : {metrics['gbt_rmse']:.3f} min")
print(f"Elapsed       : {metrics.get('total_elapsed_sec', 'N/A'):.0f}s")

---
## Section 5 — Phase 2+3: Kafka Producer + Spark Structured Streaming

Runs the producer and stream processor concurrently using Python threading.  
Bronze → Silver → Gold Parquet writes go to Google Drive.

In [ ]:
# Cell 5.1 — Start Kafka producer in background thread
import subprocess, sys, threading, os

RATE = 1000  # events/sec; try 5000 for load testing
LIMIT = 200000  # total events to replay

producer_proc = None

def run_producer():
    global producer_proc
    producer_proc = subprocess.Popen(
        [
            sys.executable, "/content/HyperLogistic_Stream/pipeline/02_kafka_producer.py",
            "--rate", str(RATE),
            "--limit", str(LIMIT),
            "--bootstrap", "localhost:9092",
            "--loop",
        ],
        cwd="/content/HyperLogistic_Stream",
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
        env={**os.environ, "HYPERLOGISTICS_CONFIG": "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml"}
    )
    for line in producer_proc.stdout:
        print(f"[PRODUCER] {line.rstrip()}")

producer_thread = threading.Thread(target=run_producer, daemon=True)
producer_thread.start()
print(f"Kafka producer started at {RATE} eps (background thread).")
print("Let it warm up for 10 seconds before starting Spark...")

import time; time.sleep(10)

In [ ]:
# Cell 5.2 — Start Spark Structured Streaming (runs until interrupted)
# The streaming job uses the Kafka-Spark connector fetched from Maven.
# This cell blocks until you interrupt it (Kernel → Interrupt).

import subprocess, sys, os

CORES = 4
SHUFFLE_PARTITIONS = 8
MAX_OFFSETS = 5000
TRIGGER_INTERVAL = 5  # seconds

SPARK_SUBMIT = "spark-submit"
KAFKA_PACKAGE = "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0"

cmd = [
    SPARK_SUBMIT,
    "--packages", KAFKA_PACKAGE,
    "--driver-memory", "16g",
    "/content/HyperLogistic_Stream/pipeline/03_stream_processor.py",
    "--cores", str(CORES),
    "--shuffle-partitions", str(SHUFFLE_PARTITIONS),
    "--max-offsets", str(MAX_OFFSETS),
    "--trigger-interval", str(TRIGGER_INTERVAL),
    "--bootstrap", "localhost:9092",
    "--benchmark-label", f"c{CORES}_sp{SHUFFLE_PARTITIONS}_mo{MAX_OFFSETS}",
]

print("Starting Spark Structured Streaming...")
print("Press Kernel → Interrupt to stop.")
print(f"Cmd: {' '.join(cmd)}")

stream_proc = subprocess.run(
    cmd,
    cwd="/content/HyperLogistic_Stream",
    env={
        **os.environ,
        "HYPERLOGISTICS_CONFIG": "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml",
    }
)

In [ ]:
# Cell 5.3 — Quick validation: check Gold table has records
import glob, pandas as pd
DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"

gold_files = glob.glob(f"{DRIVE_ROOT}/data/gold/predictions/**/*.parquet", recursive=True)
if gold_files:
    sample = pd.read_parquet(gold_files[-1])
    print(f"Gold table: {len(sample)} rows in latest partition")
    print(sample[["flight_id", "origin", "dest", "dep_delay", "predicted_delta_delay", "is_high_risk"]].head(5))
else:
    print("No Gold files yet — run the stream processor first.")

---
## Section 6 — Phase 5: FastAPI + Streamlit via ngrok

Exposes the API and dashboard publicly via ngrok tunnels.

In [ ]:
# Cell 6.1 — Start FastAPI in background
import subprocess, threading, os, sys

def run_api():
    subprocess.run(
        [sys.executable, "-m", "uvicorn", "api.main:app", "--host", "0.0.0.0", "--port", "8000"],
        cwd="/content/HyperLogistic_Stream",
        env={**os.environ, "HYPERLOGISTICS_CONFIG": "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml"}
    )

api_thread = threading.Thread(target=run_api, daemon=True)
api_thread.start()

import time; time.sleep(3)
print("FastAPI started on port 8000")

In [ ]:
# Cell 6.2 — Expose FastAPI via ngrok
from pyngrok import ngrok

# If you have a ngrok auth token, set it here:
# ngrok.set_auth_token("YOUR_TOKEN")  # free account at ngrok.com

api_tunnel = ngrok.connect(8000)
print(f"FastAPI public URL: {api_tunnel.public_url}")
print(f"Swagger UI:         {api_tunnel.public_url}/docs")

In [ ]:
# Cell 6.3 — Start Streamlit dashboard via ngrok
import subprocess, threading, os, sys
from pyngrok import ngrok

def run_streamlit():
    subprocess.run(
        [sys.executable, "-m", "streamlit", "run", "dashboard/app.py",
         "--server.port", "8501",
         "--server.headless", "true",
         "--server.address", "0.0.0.0"],
        cwd="/content/HyperLogistic_Stream",
        env={**os.environ, "HYPERLOGISTICS_CONFIG": "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml"}
    )

st_thread = threading.Thread(target=run_streamlit, daemon=True)
st_thread.start()

import time; time.sleep(5)

dash_tunnel = ngrok.connect(8501)
print(f"Streamlit dashboard: {dash_tunnel.public_url}")

---
## Section 7 — RQ2 Benchmarks

Runs a controlled factorial sweep. Use `--quick` for a 2×2×2×2 grid (~30 min),  
or remove it for the full 3×3×3×3 grid (~3–4 hours on Colab Pro).

Each configuration: starts the producer at the target rate, runs Spark for 30s (warmup + measurement), records throughput/latency.

In [ ]:
# Cell 7.1 — Run reduced 2x2x2x2 benchmark grid
import subprocess, sys, os

result = subprocess.run(
    [
        sys.executable, "/content/HyperLogistic_Stream/benchmark/run_benchmark.py",
        "--quick",  # remove for full 3x3x3x3 grid
        "--bootstrap", "localhost:9092",
    ],
    cwd="/content/HyperLogistic_Stream",
    env={**os.environ, "HYPERLOGISTICS_CONFIG": "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml"}
)
print(f"Benchmark runner exit code: {result.returncode}")

In [ ]:
# Cell 7.2 — Show benchmark summary table
import json
from pathlib import Path
import pandas as pd
DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"

summary_csv = f"{DRIVE_ROOT}/benchmark/results/benchmark_summary.csv"
if Path(summary_csv).exists():
    df = pd.read_csv(summary_csv)
    display(df.sort_values("avg_throughput_rps", ascending=False)
            [["label", "cores", "shuffle_partitions", "ingestion_rate",
              "avg_throughput_rps", "p50_batch_ms", "p95_batch_ms",
              "median_p50_latency_ms", "median_p95_latency_ms"]]
            .round(1))
else:
    print("No summary yet — run Cell 7.1 first.")

---
## Section 8 — Analysis Figures (RQ1 + RQ2)

In [ ]:
# Cell 8.1 — Generate RQ2 publication figures
import subprocess, sys, os

subprocess.run(
    [sys.executable, "/content/HyperLogistic_Stream/benchmark/analyze_results.py"],
    cwd="/content/HyperLogistic_Stream",
    env={**os.environ, "HYPERLOGISTICS_CONFIG": "/content/HyperLogistic_Stream/config/pipeline_config_colab.yaml"}
)

In [ ]:
# Cell 8.2 — Display all figures
from IPython.display import Image, display
from pathlib import Path

figs_dir = Path("/content/HyperLogistic_Stream/reports/figures")
for fig in sorted(figs_dir.glob("*.png")):
    print(f"\n--- {fig.name} ---")
    display(Image(str(fig), width=950))

In [ ]:
# Cell 8.3 — Copy figures to Google Drive for the report
import shutil
from pathlib import Path
DRIVE_ROOT = "/content/drive/MyDrive/HyperLogistics"

src = Path("/content/HyperLogistic_Stream/reports/figures")
dst = Path(f"{DRIVE_ROOT}/reports/figures")
dst.mkdir(parents=True, exist_ok=True)

for fig in src.glob("*.png"):
    shutil.copy2(fig, dst / fig.name)
    print(f"Copied {fig.name}")

print(f"\nAll figures saved to {dst}")

---
## Notes & Troubleshooting

### Kafka not starting
Rerun Cell 0.2 — Colab sometimes kills background processes after idle time.  
Check if Zookeeper is running: `!ps aux | grep zookeeper`

### Spark OOM on training
Reduce years or use `--skip-rf` flag in Cell 4.3.  
Switch to Colab Pro+ for 80 GB RAM.

### Drive I/O bottleneck during streaming
Checkpoints are written to `/content/checkpoints` (local SSD).  
Only Bronze/Silver/Gold Parquet go to Drive — acceptable latency.

### spark-submit not found
```python
import os
os.environ['PATH'] += ':/usr/local/lib/python3.10/dist-packages/pyspark/bin'
```

### ngrok tunnel expired (8-hour limit on free tier)
Rerun Cells 6.2 and 6.3 to get a fresh URL.